# Evaluation: Benchmarks, Evals, LM Harness Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: A Minimal Eval Framework

Define the core abstractions. An eval case has an input, an expected output, and an optional metadata dict. A scorer takes a prediction and a reference and returns a score between 0 and 1.

In [ ]:
```python

import json

from collections import Counter

class EvalCase:

    def __init__(self, input_text, expected, metadata=None):

        self.input_text = input_text

        self.expected = expected

        self.metadata = metadata or {}

class EvalSuite:

    def __init__(self, name, cases, scorers):

        self.name = name

        self.cases = cases

        self.scorers = scorers

    def run(self, model_fn):

        results = []

        for case in self.cases:

            prediction = model_fn(case.input_text)

            scores = {}

            for scorer_name, scorer_fn in self.scorers.items():

                scores[scorer_name] = scorer_fn(prediction, case.expected)

            results.append({

                "input": case.input_text,

                "expected": case.expected,

                "prediction": prediction,

                "scores": scores,

            })

        return results

In [ ]:
```

### Step 2: Scoring Functions

Build exact match, token F1, and a simulated LLM-as-judge scorer.

In [ ]:
```python

def exact_match(prediction, expected):

    return 1.0 if prediction.strip().lower() == expected.strip().lower() else 0.0

def token_f1(prediction, expected):

    pred_tokens = set(prediction.lower().split())

    exp_tokens = set(expected.lower().split())

    if not pred_tokens or not exp_tokens:

        return 0.0

    common = pred_tokens & exp_tokens

    precision = len(common) / len(pred_tokens)

    recall = len(common) / len(exp_tokens)

    if precision + recall == 0:

        return 0.0

    return 2 * (precision * recall) / (precision + recall)

def llm_judge_simulated(prediction, expected):

    pred_words = set(prediction.lower().split())

    exp_words = set(expected.lower().split())

    if not exp_words:

        return 0.0

    overlap = len(pred_words & exp_words) / len(exp_words)

    length_penalty = min(1.0, len(prediction) / max(len(expected), 1))

    return round(overlap * 0.7 + length_penalty * 0.3, 3)

In [ ]:
```

### Step 3: ELO Rating System

Implement pairwise comparisons with ELO updates. This is exactly the system Chatbot Arena uses to rank models.

In [ ]:
```python

class ELOTracker:

    def __init__(self, k=32, initial_rating=1500):

        self.ratings = {}

        self.k = k

        self.initial_rating = initial_rating

        self.history = []

    def _ensure_player(self, name):

        if name not in self.ratings:

            self.ratings[name] = self.initial_rating

    def expected_score(self, rating_a, rating_b):

        return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

    def record_match(self, player_a, player_b, outcome):

        self._ensure_player(player_a)

        self._ensure_player(player_b)

        ea = self.expected_score(self.ratings[player_a], self.ratings[player_b])

        eb = 1 - ea

        if outcome == "a":

            sa, sb = 1.0, 0.0

        elif outcome == "b":

            sa, sb = 0.0, 1.0

        else:

            sa, sb = 0.5, 0.5

        self.ratings[player_a] += self.k * (sa - ea)

        self.ratings[player_b] += self.k * (sb - eb)

        self.history.append({

            "a": player_a, "b": player_b,

            "outcome": outcome,

            "rating_a": round(self.ratings[player_a], 1),

            "rating_b": round(self.ratings[player_b], 1),

        })

    def leaderboard(self):

        return sorted(self.ratings.items(), key=lambda x: -x[1])

In [ ]:
```

### Step 4: Perplexity Calculation

Compute perplexity using token probabilities. In practice you would get these from the model's logits. Here we simulate with a probability distribution.

In [ ]:
```python

import numpy as np

def perplexity(log_probs):

    if not log_probs:

        return float("inf")

    avg_neg_log_prob = -np.mean(log_probs)

    return float(np.exp(avg_neg_log_prob))

def token_log_probs_simulated(text, model_quality=0.8):

    np.random.seed(hash(text) % 2**31)

    tokens = text.split()

    log_probs = []

    for i, token in enumerate(tokens):

        base_prob = model_quality

        if len(token) > 8:

            base_prob *= 0.6

        if i == 0:

            base_prob *= 0.7

        prob = np.clip(base_prob + np.random.normal(0, 0.1), 0.01, 0.99)

        log_probs.append(float(np.log(prob)))

    return log_probs

In [ ]:
```

### Step 5: Aggregate Results

Compute summary statistics across an eval run: mean, median, pass rate at a threshold, and per-metric breakdowns.

In [ ]:
```python

def summarize_results(results, threshold=0.8):

    all_scores = {}

    for r in results:

        for metric, score in r["scores"].items():

            all_scores.setdefault(metric, []).append(score)

    summary = {}

    for metric, scores in all_scores.items():

        arr = np.array(scores)

        summary[metric] = {

            "mean": round(float(np.mean(arr)), 3),

            "median": round(float(np.median(arr)), 3),

            "std": round(float(np.std(arr)), 3),

            "min": round(float(np.min(arr)), 3),

            "max": round(float(np.max(arr)), 3),

            "pass_rate": round(float(np.mean(arr >= threshold)), 3),

            "n": len(scores),

        }

    return summary

def print_summary(summary, suite_name="Eval"):

    print(f"\n{'=' * 60}")

    print(f"  {suite_name} Summary")

    print(f"{'=' * 60}")

    for metric, stats in summary.items():

        print(f"\n  {metric}:")

        print(f"    Mean:      {stats['mean']:.3f}")

        print(f"    Median:    {stats['median']:.3f}")

        print(f"    Std:       {stats['std']:.3f}")

        print(f"    Range:     [{stats['min']:.3f}, {stats['max']:.3f}]")

        print(f"    Pass rate: {stats['pass_rate']:.1%} (threshold >= 0.8)")

        print(f"    N:         {stats['n']}")

In [ ]:
```

### Step 6: Run the Full Pipeline

Wire everything together. Define a task, create test cases, simulate two models, run evals, compute ELO from pairwise comparisons, and print the leaderboard.

In [ ]:
```python

def demo_model_good(prompt):

    responses = {

        "What is the capital of France?": "Paris",

        "What is 2 + 2?": "4",

        "Who wrote Hamlet?": "William Shakespeare",

        "What language is PyTorch written in?": "Python and C++",

        "What is the boiling point of water?": "100 degrees Celsius",

    }

    return responses.get(prompt, "I don't know")

def demo_model_bad(prompt):

    responses = {

        "What is the capital of France?": "Paris is the capital city of France",

        "What is 2 + 2?": "The answer is four",

        "Who wrote Hamlet?": "Shakespeare",

        "What language is PyTorch written in?": "Python",

        "What is the boiling point of water?": "212 Fahrenheit",

    }

    return responses.get(prompt, "Unknown")

cases = [

    EvalCase("What is the capital of France?", "Paris"),

    EvalCase("What is 2 + 2?", "4"),

    EvalCase("Who wrote Hamlet?", "William Shakespeare"),

    EvalCase("What language is PyTorch written in?", "Python and C++"),

    EvalCase("What is the boiling point of water?", "100 degrees Celsius"),

]

suite = EvalSuite(

    name="General Knowledge",

    cases=cases,

    scorers={

        "exact_match": exact_match,

        "token_f1": token_f1,

        "llm_judge": llm_judge_simulated,

    },

)

results_good = suite.run(demo_model_good)

results_bad = suite.run(demo_model_bad)

print_summary(summarize_results(results_good), "Model A (concise)")

print_summary(summarize_results(results_bad), "Model B (verbose)")

In [ ]:
```

The "good" model gives exact answers. The "bad" model gives verbose paraphrases. Exact match punishes the verbose model severely. Token F1 and LLM-as-judge are more forgiving. This illustrates why metric choice matters: the same model looks great or terrible depending on how you score it.

### Step 7: ELO Tournament

Run pairwise comparisons between models across multiple rounds.

In [ ]:
```python

elo = ELOTracker(k=32)

for case in cases:

    pred_a = demo_model_good(case.input_text)

    pred_b = demo_model_bad(case.input_text)

    score_a = token_f1(pred_a, case.expected)

    score_b = token_f1(pred_b, case.expected)

    if score_a > score_b:

        outcome = "a"

    elif score_b > score_a:

        outcome = "b"

    else:

        outcome = "tie"

    elo.record_match("model_a_concise", "model_b_verbose", outcome)

print("\nELO Leaderboard:")

for name, rating in elo.leaderboard():

    print(f"  {name}: {rating:.0f}")

In [ ]:
```

### Step 8: Perplexity Comparison

Compare perplexity across "models" of different quality levels.

In [ ]:
```python

test_text = "The quick brown fox jumps over the lazy dog in the garden"

for quality, label in [(0.9, "Strong model"), (0.7, "Medium model"), (0.4, "Weak model")]:

    log_probs = token_log_probs_simulated(test_text, model_quality=quality)

    ppl = perplexity(log_probs)

    print(f"  {label} (quality={quality}): perplexity = {ppl:.2f}")

In [ ]:
```

## Exercises

In [ ]:
1. Add a "consistency" scorer that runs the same input through the model 5 times and measures how often the outputs match. Inconsistent answers on deterministic inputs reveal fragile prompts or high temperature settings.

2. Extend the ELO tracker to support multiple judge functions (exact match, F1, LLM-as-judge) and weight them. Compare how the leaderboard changes when you weight exact match heavily versus F1 heavily.

3. Build an eval suite for a specific task: email classification into 5 categories. Create 100 test cases with diverse examples including edge cases (emails that could belong to multiple categories, empty emails, emails in other languages). Measure how different "models" (rule-based, keyword matching, simulated LLM) perform.

4. Implement contamination detection: given a set of eval questions and a training corpus, check what percentage of eval questions (or close paraphrases) appear in the training data. This is how researchers audit benchmark validity.

5. Build a "model diff" tool. Given eval results from two model versions, highlight which specific test cases improved, which regressed, and which stayed the same. This is the eval equivalent of a code diff -- essential for understanding whether a change helped or hurt.